# DIAMOND read hits: filtering and best hit

Post-processing of the DIAMOND blastx results produced by `run_diamond_reads.sh`, for every database (UNIREF, DB1-DB6).

Files are read in chunks (`CHUNKSIZE` rows at a time), since read-level outputs can be very large.

**Step 1 – Filter hits.** Reads `DIAMOND_<DB>/<sample>.tsv`, computes subject coverage and keeps hits that meet all cut-offs. Output: `DIAMOND_<DB>/<DB>_filtered/<sample>_filtered.tsv`.

- `scovhsp` = alignment length / slen × 100 (subject coverage; DIAMOND only reports query coverage, `qcovhsp`)

| Metric | Cut-off |
|---|---|
| e-value | < 1 |
| identity (`pident`) | > 50 % |
| query coverage (`qcovhsp`) | > 90 % |
| subject coverage (`scovhsp`) | > 10 % |

**Step 2 – Best hit per read.** Keeps, for each read (`qseqid`), only the hit with the highest bitscore. Output: `DIAMOND_<DB>/<DB>_best_hit/<sample>_besthit.tsv`.

In [ ]:
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------
DBS = ["UNIREF", "DB1", "DB2", "DB3", "DB4", "DB5", "DB6"]

COLUMNS = [
    "qseqid", "sseqid", "qlen", "slen",
    "length", "evalue", "bitscore", "qcovhsp",
    "pident", "ppos",
]

MAX_EVALUE = 1
MIN_PIDENT = 50
MIN_QCOV = 90
MIN_SCOV = 10

CHUNKSIZE = 1_000_000   # rows per chunk

## Step 1 – Filter hits

In [ ]:
for db in DBS:
    input_dir = Path(f"DIAMOND_{db}")
    filtered_dir = input_dir / f"{db}_filtered"
    filtered_dir.mkdir(parents=True, exist_ok=True)

    for file in input_dir.glob("*.tsv"):
        print(f"[{db}] Filtering {file.name}...")

        output_file = filtered_dir / f"{file.stem}_filtered.tsv"

        # remove previous output, if any (results are appended chunk by chunk)
        if output_file.exists():
            output_file.unlink()

        reader = pd.read_csv(
            file,
            sep="\t",
            header=None,
            names=COLUMNS,
            chunksize=CHUNKSIZE,
            low_memory=True,
        )

        for i, chunk in enumerate(reader):
            chunk["scovhsp"] = chunk["length"] / chunk["slen"] * 100

            filtered = chunk[
                (chunk["evalue"] < MAX_EVALUE) &
                (chunk["pident"] > MIN_PIDENT) &
                (chunk["qcovhsp"] > MIN_QCOV) &
                (chunk["scovhsp"] > MIN_SCOV)
            ]

            filtered.to_csv(
                output_file,
                sep="\t",
                index=False,
                mode="a",
                header=(i == 0),
            )

            print(f"  Chunk {i + 1} done.")

print("Filtering done!")

## Step 2 – Best hit per read

In [ ]:
for db in DBS:
    filtered_dir = Path(f"DIAMOND_{db}") / f"{db}_filtered"
    best_hit_dir = Path(f"DIAMOND_{db}") / f"{db}_best_hit"
    best_hit_dir.mkdir(parents=True, exist_ok=True)

    for file in filtered_dir.glob("*_filtered.tsv"):
        print(f"[{db}] Best hit {file.name}...")

        best_hits = pd.DataFrame()

        reader = pd.read_csv(
            file,
            sep="\t",
            chunksize=CHUNKSIZE,
            low_memory=True,
        )

        for i, chunk in enumerate(reader):
            # best hit of each qseqid within the chunk
            chunk = (
                chunk.sort_values("bitscore", ascending=False)
                     .drop_duplicates(subset="qseqid", keep="first")
            )

            # merge with the best hits found in previous chunks
            best_hits = pd.concat([best_hits, chunk], ignore_index=True)

            # keep only the highest bitscore per qseqid
            best_hits = (
                best_hits.sort_values("bitscore", ascending=False)
                         .drop_duplicates(subset="qseqid", keep="first")
            )

            print(f"  Chunk {i + 1} done.")

        best_hits = best_hits.sort_values("qseqid")

        output_file = best_hit_dir / file.name.replace("_filtered.tsv", "_besthit.tsv")
        best_hits.to_csv(output_file, sep="\t", index=False)

print("Best hit selection done!")